# 03 — Member Master Build

## Objective
Build the first core analytical table of the project:

`member_master.csv`

This table combines:
- insured member profile
- policy information

and becomes the base for:
- portfolio overview
- first segmentation
- early risk scoring work
- pricing exploration

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\dolivares\Desktop\novares-securehealth


In [2]:
import pandas as pd
import numpy as np

from src.config import PROCESSED_DIR
from src.data.load_data import load_insured_members, load_policies

In [3]:
## Load source tables
members = load_insured_members()
policies = load_policies()

print("members shape:", members.shape)
print("policies shape:", policies.shape)
members = load_insured_members()
policies = load_policies()

print("members shape:", members.shape)
print("policies shape:", policies.shape)

members shape: (4000, 32)
policies shape: (4000, 25)
members shape: (4000, 32)
policies shape: (4000, 25)


In [4]:
display(members.head(3))
display(policies.head(3))

,member_id,policy_id,join_date,age,age_band,sex,region,city_tier,socioeconomic_band,employment_status,...,self_management_adherence,archetype,baseline_risk_score,utilization_propensity,acute_event_propensity,misuse_exposure_propensity,price_sensitivity,coverage_preference,network_preference,preferred_plan_type
0,MBR000001,POL000001,2022-09-24,47,45-54,M,San Pedro,Urban,Upper-Middle,Self-Employed,...,Low,Hyper-Utilizer / Misuse Risk,0.591,0.847,0.205,0.865,High,Balanced,Balanced,Managed Review
1,MBR000002,POL000002,2024-04-02,35,35-44,F,San Pedro,Metro,Lower-Middle,Employed,...,Low,Healthy Low User,0.156,0.104,0.074,0.085,Medium,Basic,Narrow,Essential
2,MBR000003,POL000003,2024-06-27,65,65+,M,Asunción,Urban,Middle,Self-Employed,...,High,Chronic Managed,0.728,0.776,0.223,0.056,Medium,Broad,Broad,Chronic Care


,policy_id,member_id,policy_start_date,policy_end_date,active_flag,plan_type,plan_tier,coverage_scope,provider_network_type,deductible_amount,...,premium_monthly,premium_annual,underwriting_load_factor,discount_factor,recommended_plan_flag,pricing_adequacy_ratio,renewal_flag,cancellation_flag,sales_channel,broker_id
0,POL000001,MBR000001,2022-09-24,2026-12-31,1,Managed Review,Mid,Full,Balanced,199.02,...,291.4,3496.8,1.139,0.092,1,0.851,1,0,Corporate,NaN
1,POL000002,MBR000002,2024-04-02,2026-12-31,1,Essential,Basic,Ambulatory,Narrow,597.15,...,198.7,2384.4,1.138,0.082,1,1.285,1,0,Direct,NaN
2,POL000003,MBR000003,2024-06-27,2026-12-31,1,Chronic Care,Premium,Full,Broad,196.63,...,430.1,5161.2,1.005,0.060,1,0.994,0,0,Direct,NaN


In [5]:
## Pre-merge validation
validation = pd.DataFrame([
    {
        "check": "members.member_id unique",
        "result": members["member_id"].nunique() == len(members)
    },
    {
        "check": "members.policy_id unique",
        "result": members["policy_id"].nunique() == len(members)
    },
    {
        "check": "policies.member_id unique",
        "result": policies["member_id"].nunique() == len(policies)
    },
    {
        "check": "policies.policy_id unique",
        "result": policies["policy_id"].nunique() == len(policies)
    },
    {
        "check": "same member_id universe",
        "result": set(members["member_id"]) == set(policies["member_id"])
    },
    {
        "check": "same policy_id universe",
        "result": set(members["policy_id"]) == set(policies["policy_id"])
    },
])

validation

,check,result
0,members.member_id unique,True
1,members.policy_id unique,True
2,policies.member_id unique,True
3,policies.policy_id unique,True
4,same member_id universe,True
5,same policy_id universe,True


In [6]:
## Merge members + policies
member_master = members.merge(
    policies,
    on=["member_id", "policy_id"],
    how="left",
    validate="one_to_one"
)

print("member_master shape:", member_master.shape)
member_master.head(3)

member_master shape: (4000, 55)


,member_id,policy_id,join_date,age,age_band,sex,region,city_tier,socioeconomic_band,employment_status,...,premium_monthly,premium_annual,underwriting_load_factor,discount_factor,recommended_plan_flag,pricing_adequacy_ratio,renewal_flag,cancellation_flag,sales_channel,broker_id
0,MBR000001,POL000001,2022-09-24,47,45-54,M,San Pedro,Urban,Upper-Middle,Self-Employed,...,291.4,3496.8,1.139,0.092,1,0.851,1,0,Corporate,NaN
1,MBR000002,POL000002,2024-04-02,35,35-44,F,San Pedro,Metro,Lower-Middle,Employed,...,198.7,2384.4,1.138,0.082,1,1.285,1,0,Direct,NaN
2,MBR000003,POL000003,2024-06-27,65,65+,M,Asunción,Urban,Middle,Self-Employed,...,430.1,5161.2,1.005,0.060,1,0.994,0,0,Direct,NaN


In [7]:
## Post-merge validation
post_merge_checks = pd.DataFrame([
    {
        "check": "rows preserved from members",
        "result": len(member_master) == len(members)
    },
    {
        "check": "member_id unique",
        "result": member_master["member_id"].nunique() == len(member_master)
    },
    {
        "check": "policy_id unique",
        "result": member_master["policy_id"].nunique() == len(member_master)
    }
])

post_merge_checks

,check,result
0,rows preserved from members,True
1,member_id unique,True
2,policy_id unique,True


In [8]:
## Create first derived variables
member_master["policy_start_date"] = pd.to_datetime(member_master["policy_start_date"], errors="coerce")
member_master["policy_end_date"] = pd.to_datetime(member_master["policy_end_date"], errors="coerce")
member_master["join_date"] = pd.to_datetime(member_master["join_date"], errors="coerce")

member_master["tenure_days"] = (member_master["policy_end_date"] - member_master["join_date"]).dt.days
member_master["tenure_months_approx"] = (member_master["tenure_days"] / 30.4).round(1)

member_master["premium_gap"] = member_master["premium_annual"] - (member_master["premium_monthly"] * 12)
member_master["has_dependents_flag"] = (member_master["dependents_n"] > 0).astype(int)
member_master["high_chronic_burden_flag"] = (member_master["chronic_condition_count"] >= 2).astype(int)
member_master["high_baseline_risk_flag"] = (member_master["baseline_risk_score"] >= member_master["baseline_risk_score"].quantile(0.80)).astype(int)
member_master["high_utilization_propensity_flag"] = (member_master["utilization_propensity"] >= member_master["utilization_propensity"].quantile(0.80)).astype(int)
member_master["high_misuse_exposure_flag"] = (member_master["misuse_exposure_propensity"] >= member_master["misuse_exposure_propensity"].quantile(0.80)).astype(int)

In [9]:
derived_cols = [
    "tenure_days",
    "tenure_months_approx",
    "premium_gap",
    "has_dependents_flag",
    "high_chronic_burden_flag",
    "high_baseline_risk_flag",
    "high_utilization_propensity_flag",
    "high_misuse_exposure_flag",
]

member_master[["member_id", "policy_id"] + derived_cols].head(10)

,member_id,policy_id,tenure_days,tenure_months_approx,premium_gap,has_dependents_flag,high_chronic_burden_flag,high_baseline_risk_flag,high_utilization_propensity_flag,high_misuse_exposure_flag
0,MBR000001,POL000001,1559,51.3,4.547474e-13,0,0,1,1,1
1,MBR000002,POL000002,1003,33.0,4.547474e-13,0,0,0,0,0
2,MBR000003,POL000003,917,30.2,-9.094947e-13,1,0,1,1,0
3,MBR000004,POL000004,951,31.3,0.000000e+00,0,0,0,0,0
4,MBR000005,POL000005,1228,40.4,9.094947e-13,0,0,1,0,1
5,MBR000006,POL000006,1268,41.7,0.000000e+00,1,0,1,1,0
6,MBR000007,POL000007,1610,53.0,0.000000e+00,1,0,0,0,1
7,MBR000008,POL000008,1512,49.7,4.547474e-13,1,0,0,0,1
8,MBR000009,POL000009,1016,33.4,0.000000e+00,0,0,1,0,0
9,MBR000010,POL000010,1239,40.8,0.000000e+00,0,0,0,0,0


In [10]:
## Quick profiling of member master
profile_summary = {
    "rows": len(member_master),
    "columns": member_master.shape[1],
    "active_policies_n": int(member_master["active_flag"].sum()),
    "renewal_rate": round(member_master["renewal_flag"].mean(), 4),
    "cancellation_rate": round(member_master["cancellation_flag"].mean(), 4),
    "avg_monthly_premium": round(member_master["premium_monthly"].mean(), 2),
    "avg_baseline_risk_score": round(member_master["baseline_risk_score"].mean(), 3),
    "avg_utilization_propensity": round(member_master["utilization_propensity"].mean(), 3),
}

pd.DataFrame(profile_summary.items(), columns=["metric", "value"])

,metric,value
0,rows,4000.0000
1,columns,63.0000
2,active_policies_n,4000.0000
3,renewal_rate,0.8468
4,cancellation_rate,0.0585
5,avg_monthly_premium,382.7200
6,avg_baseline_risk_score,0.3030
7,avg_utilization_propensity,0.3660


In [11]:
display(
    member_master["archetype"]
    .value_counts(dropna=False)
    .rename_axis("archetype")
    .reset_index(name="count")
)

display(
    member_master["plan_type"]
    .value_counts(dropna=False)
    .rename_axis("plan_type")
    .reset_index(name="count")
)

,archetype,count
0,Healthy Low User,1214
1,Preventive User,867
2,Family Planner,719
3,Chronic Managed,708
4,High Acute Risk,321
5,Hyper-Utilizer / Misuse Risk,171


,plan_type,count
0,Essential,1214
1,Standard,867
2,Family Plus,719
3,Chronic Care,708
4,High Protection,321
5,Managed Review,171


In [12]:
## Save analytical base
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_file = PROCESSED_DIR / "member_master.csv"
member_master.to_csv(output_file, index=False)

print("Saved:", output_file)

Saved: C:\Users\dolivares\Desktop\novares-securehealth\data\processed\member_master.csv


In [13]:
## Final notes
notes = [
    "member_master.csv is now the first processed analytical table of the repository.",
    "This table can already feed Portfolio Overview and first business segmentation.",
    "Next analytical priority should be building claims_analytical_base.csv.",
]

for i, note in enumerate(notes, 1):
    print(f"{i}. {note}")

1. member_master.csv is now the first processed analytical table of the repository.
2. This table can already feed Portfolio Overview and first business segmentation.
3. Next analytical priority should be building claims_analytical_base.csv.
